# СБП в `ods.scd1_z_main_docum`: ESPP-документы (для Алины)

Исходный скрипт коллеги:

```sql
select * from ods.scd1_z_main_docum md
join ods.scd1_z_name_paydoc np on np.id = md.c_vid_doc and np.c_code = 'БЕЗН_ПЛ_ПОРУЧ'
join eq_account ea on ea.id = md.c_acc_kt
where 1=1
  and md.c_document_uno like 'ESPP%'
```

`eq_account` (ответ автора, «остаток от ЦТЕ, можно пока без неё»):

```sql
with eq_account as (
  select af.id, af.c_main_v_id
  from ods.scd1_z_r2_ip_merchants m
  join ods.scd1_z_r2_ip_acc a on a.c_parent_id = m.id
  join ods.scd1_z_ac_fin af on af.id = a.c_dt_1_2
)
```

Что выяснили на первом прогоне (июль 2026, полный скан):
- `ESPP` — система приёма платежей, а не только СБП. В ESPP-платёжках 82% документов со словами СБП / НСПК,
  остальное — оплата связи через агрегаторов (ПСБ, Элекснет, «Город») и комиссии банка.
- 98% суммы платёжек — проводки 40817 → 30232: исходящие платежи клиентов банка.
- ESPP есть не только в платёжных поручениях: банковские ордера (4,8 млн), мемориальные (2,1 млн), кассовые (0,1 млн).

Как устроена тетрадка:
1. `main_docum` читается **один раз**: ESPP-документы всех видов за период сохраняются во временную таблицу в `sandbox_ai`.
   Дальше все запросы идут по ней и занимают секунды.
2. **Быстрый режим**: `SAMPLE_PCT = 1` читает ~1% файлов (`tablesample`). Для итоговых цифр — `SAMPLE_PCT = None`.
3. Каждый документ получает две метки: **вид операции** (`kind`, по назначению платежа) и **направление**
   (`direction`, по балансовым счетам). Правила — в одной ячейке; из них же собирается итоговый SQL для Алины.

Шаги:
1. Партиции и формат `main_docum`.
2. Временная таблица ESPP-документов.
3. Разрез по видам документов и направлениям.
4. Классификация: вид операции × направление.
5. Неразмеченное (`kind = other`) — чтобы дописать правила.
6. Банковские, мемориальные и кассовые ордера: что в них.
7. Что оставляет `join eq_account` — по ответу автора, это счета мерчантов (CTE из ЦТЕ).
8. Итог для Алины + готовые SQL-файлы.
9. Выгрузка в Excel.


In [ ]:
import re
import sys
import time
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 300)

LAKE_USER = 'Shestopalov-VYur'
LAKE_PASSWORD = None
for cand in [
    Path.cwd(),
    Path.cwd() / 'sources' / 'sql',
    Path('/home/jovyan/documents/Equaring'),
    Path('/home/jovyan/documents/Equaring/sources/sql'),
]:
    if (cand / 'connection_secrets.py').exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        from connection_secrets import LAKE_USER, LAKE_PASSWORD
        break

user_params = {'user_name': LAKE_USER}
if LAKE_PASSWORD:
    user_params['password'] = LAKE_PASSWORD

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params=user_params,
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
# Период по c_date_prov: [PERIOD_START, PERIOD_END_EXCL)
PERIOD_START = '2026-07-01'
PERIOD_END_EXCL = '2026-08-01'

# Процент файлов main_docum для tablesample. None — полный прогон для итоговых цифр
SAMPLE_PCT = 1

# False — не пересобирать временную таблицу, если она уже есть
REBUILD = True

MEM_LIMIT = '16g'

MAIN_DOCUM = 'ods.scd1_z_main_docum'
NAME_PAYDOC = 'ods.scd1_z_name_paydoc'
UNO_PREFIX = 'ESPP'
PAYDOC_CODE = 'БЕЗН_ПЛ_ПОРУЧ'

# eq_account из скрипта коллеги — CTE из ЦТЕ (ответ автора): счета мерчантов
EQ_ACCOUNT_CTE = '''eq_account as (
  select af.id, af.c_main_v_id
  from ods.scd1_z_r2_ip_merchants m
  join ods.scd1_z_r2_ip_acc a on a.c_parent_id = m.id
  join ods.scd1_z_ac_fin af on af.id = a.c_dt_1_2
)'''

TMP_DB = 'sandbox_ai'
TMP_NAME = 'tmp_sbp_espp_' + PERIOD_START[:7].replace('-', '') + (f'_s{SAMPLE_PCT}' if SAMPLE_PCT else '_full')
TMP_TABLE = f'{TMP_DB}.{TMP_NAME}'

OUT_XLSX = './sbp_espp_main_docum.xlsx'
OUT_SQL_AGG = './sbp_espp_final_agg.sql'
OUT_SQL_DETAIL = './sbp_espp_final_detail.sql'

# Без cast: строковое сравнение даёт Impala пропускать блоки Parquet по min/max c_date_prov
PERIOD_WHERE = f"md.c_date_prov >= '{PERIOD_START}' and md.c_date_prov < '{PERIOD_END_EXCL}'"

# Impala lower() не работает с кириллицей, поэтому варианты регистра перечислены явно
SBP_NAZN_RLIKE = '(СБП|сбп|SBP|sbp|[Бб]ыстрых платеж|БЫСТРЫХ ПЛАТЕЖ|НСПК|NSPK)'

SCALE = 100 / SAMPLE_PCT if SAMPLE_PCT else 1


def q(sql):
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    df.columns = [str(c).lower() for c in df.columns]
    return df


def ex(sql):
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        imp.execute(sql)


# Коннектор отдаёт числа как object (str / Decimal). Приводим только счётчики и суммы:
# номера счетов и id нельзя гнать через to_numeric — 20 знаков не влезают во float
def num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


def add_share(df, col='docs'):
    df['share'] = (df[col] / df[col].sum()).round(4)
    return df


results = {}
print('period:', PERIOD_START, '..', PERIOD_END_EXCL)
print('mode:', f'tablesample {SAMPLE_PCT}% (цифры × {SCALE:g} — грубая оценка)' if SAMPLE_PCT else 'полный прогон')
print('tmp table:', TMP_TABLE)


## Правила классификации

**Вид операции (`kind`)** — по назначению платежа, первое сработавшее правило. **Направление (`direction`)** — по первым
5 цифрам счетов дебета и кредита. Порядок важен: комиссия проверяется раньше, чем слова «СБП».

Счета, на которые опираются направления:
- 40817 / 40820 и другие 405–408 — счета клиентов (физлица, юрлица, ИП);
- 30232 / 30233 — незавершённые расчёты с операторами платёжной инфраструктуры и операторами по переводу (в т.ч. НСПК);
- 40911 — транзитные переводы без открытия счёта;
- 30102 / 30301 / 30109 — корсчёт в ЦБ, межфилиальные расчёты, счета банков-корреспондентов;
- 47423 — доходы банка к получению (комиссия).

Если в шаге 5 найдутся крупные неразмеченные шаблоны — допишите правило сюда и перезапустите шаги 4–8
(временную таблицу пересобирать не нужно).


In [ ]:
KIND_RULES = [
    ('commission', 'Комиссия банка', "bal_kt = '47423' or c_nazn rlike 'Комиссия [Бб]анка'"),
    ('sbp_c2b', 'Покупка у компании через СБП', "c_nazn rlike 'Оплата товаров и услуг.*через СБП'"),
    ('sbp_qr', 'Оплата по QR-коду', "c_nazn rlike 'по QR-коду'"),
    ('sbp_nesp', 'Пополнение кошелька (НЭСП) через СБП', "c_nazn rlike 'НЭСП.*СБП'"),
    ('sbp_other', 'Прочее со словами СБП / НСПК', f"c_nazn rlike '{SBP_NAZN_RLIKE}'"),
    ('services', 'Оплата услуг через агрегатор (не СБП)', "c_nazn rlike 'за услугу'"),
    ('agent_registry', 'Сбор платежей по реестрам агентов', "c_nazn rlike 'Сбор платежей'"),
]
SBP_KINDS = ['sbp_c2b', 'sbp_qr', 'sbp_nesp', 'sbp_other']

CLIENT_ACC = '^(40[5-8]|423|426)'
DIRECTION_RULES = [
    ('fee', 'Комиссия в доход банка', "bal_kt = '47423'"),
    ('out', 'Со счёта клиента в расчёты (исходящий)',
     f"bal_kt in ('30232', '30233') and (bal_dt rlike '{CLIENT_ACC}' or bal_dt = '40911')"),
    ('in', 'Из расчётов на счёт клиента (входящий)',
     f"bal_dt in ('30232', '30233', '40911') and bal_kt rlike '{CLIENT_ACC}'"),
    ('settlement', 'Расчёты между банками / с НСПК', "bal_dt rlike '^(30|40911|4742)' and bal_kt rlike '^(30|4742)'"),
]
# Только эти направления — клиентские операции; settlement и fee дублируют их деньги
COUNTED_DIRECTIONS = ['out', 'in']

KIND_LABEL = {c: l for c, l, _ in KIND_RULES} | {'other': 'Не размечено'}
DIRECTION_LABEL = {c: l for c, l, _ in DIRECTION_RULES} | {'other': 'Прочие пары счетов'}


def case_sql(rules, default='other'):
    whens = ''.join(f"    when {cond} then '{c}'\n" for c, _, cond in rules)
    return f"case\n{whens}    else '{default}'\n  end"


def espp_base_sql(sample_pct=None):
    sample = f' tablesample system({sample_pct})' if sample_pct else ''
    return f'''select
  md.id as doc_id,
  md.c_document_uno,
  np.c_code as vid_code,
  substr(cast(md.c_date_prov as string), 1, 10) as d_prov,
  cast(md.c_sum as decimal(38,2)) as c_sum,
  md.c_num_dt,
  md.c_num_kt,
  md.c_acc_dt,
  md.c_acc_kt,
  substr(md.c_num_dt, 1, 5) as bal_dt,
  substr(md.c_num_kt, 1, 5) as bal_kt,
  md.c_nazn
from {MAIN_DOCUM} md{sample}
left join {NAME_PAYDOC} np on np.id = md.c_vid_doc
where md.c_document_uno like '{UNO_PREFIX}%'
  and {PERIOD_WHERE}'''


def classify_sql(src_sql):
    return f'''select
  t.*,
  {case_sql(KIND_RULES)} as kind,
  {case_sql(DIRECTION_RULES)} as direction
from (
{src_sql}
) t'''


CLS = classify_sql(f'select * from {TMP_TABLE}')
print(CLS)


## 1) Партиции и формат `main_docum`

Если таблица разбита на партиции по дате, фильтр по полю партиции — самый дешёвый способ сузить чтение.
`show create table` показывает `PARTITIONED BY` и формат хранения (`STORED AS PARQUET` / Kudu / Iceberg).
Для непартиционированной таблицы `show partitions` либо падает, либо отдаёт одну строку `Total` с числом файлов и размером.


In [ ]:
try:
    ddl = q(f'show create table {MAIN_DOCUM}')
    ddl_text = '\n'.join(ddl.iloc[:, 0].astype(str))
    keep = [ln for ln in ddl_text.splitlines() if re.search(r'PARTITION|STORED|LOCATION|TBLPROPERTIES|kudu|iceberg', ln, re.I)]
    print('\n'.join(keep) or ddl_text[:3000])
except Exception as e:
    print('show create table:', type(e).__name__, str(e)[:300])

try:
    parts = q(f'show partitions {MAIN_DOCUM}')
    print('\nПартиций (строк):', len(parts))
    display(parts.head(5))
    display(parts.tail(5))
except Exception as e:
    print('\nshow partitions:', type(e).__name__, str(e)[:300])


## 2) Временная таблица ESPP-документов

Единственный тяжёлый запрос. ESPP-документы **всех видов** за период + вид документа, суммы, счета, назначение.
В быстром режиме — ~1% файлов. Если `tablesample` не поддерживается (Kudu / Iceberg), строится без выборки.


In [ ]:
existing = q(f"show tables in {TMP_DB} like '{TMP_NAME}'")
if not REBUILD and not existing.empty:
    print('Беру готовую', TMP_TABLE)
else:
    ex(f'drop table if exists {TMP_TABLE}')
    t0 = time.time()
    try:
        ex(f'create table {TMP_TABLE} stored as parquet as\n{espp_base_sql(SAMPLE_PCT)}')
    except Exception as e:
        if SAMPLE_PCT and 'tablesample' in str(e).lower():
            print('tablesample не поддерживается для этой таблицы, строю без выборки:', str(e)[:200])
            SAMPLE_PCT, SCALE = None, 1
            ex(f'create table {TMP_TABLE} stored as parquet as\n{espp_base_sql(None)}')
        else:
            raise
    print(f'Построено за {time.time() - t0:.0f} c')

tmp_total = num(q(f'''
select count(*) as docs, sum(c_sum) as sum_total, min(d_prov) as d_min, max(d_prov) as d_max
from {TMP_TABLE}
'''), ['docs', 'sum_total'])
display(tmp_total)


## 3) Виды документов × направления

Где лежат ESPP-документы и куда идут деньги. На полном прогоне: платёжки 9,2 млн, банковские ордера 4,8 млн,
мемориальные 2,1 млн. Здесь видно, не дублируют ли ордера платёжки (`settlement`) или это самостоятельные операции (`in` / `out`).


In [ ]:
vid_dir = num(q(f'''
select vid_code, direction, count(*) as docs, sum(c_sum) as sum_total
from ({CLS}) x
group by 1, 2
order by docs desc
'''), ['docs', 'sum_total'])
vid_dir['direction_label'] = vid_dir['direction'].map(DIRECTION_LABEL)
add_share(vid_dir)
display(vid_dir)

display(vid_dir.pivot_table(index='vid_code', columns='direction', values='docs', aggfunc='sum', fill_value=0))
results['vid_x_direction'] = vid_dir


## 4) Классификация: вид операции × направление

`is_sbp` — вид операции относится к СБП. `counted` — направление клиентское (`out` / `in`), такие строки можно
суммировать без двойного счёта.


In [ ]:
kind_dir = num(q(f'''
select kind, direction, count(*) as docs, sum(c_sum) as sum_total
from ({CLS}) x
group by 1, 2
order by docs desc
'''), ['docs', 'sum_total'])
kind_dir['kind_label'] = kind_dir['kind'].map(KIND_LABEL)
kind_dir['direction_label'] = kind_dir['direction'].map(DIRECTION_LABEL)
kind_dir['is_sbp'] = kind_dir['kind'].isin(SBP_KINDS)
kind_dir['counted'] = kind_dir['direction'].isin(COUNTED_DIRECTIONS)
add_share(kind_dir)
display(kind_dir)

print('Документов, шт.:')
display(kind_dir.pivot_table(index='kind_label', columns='direction', values='docs', aggfunc='sum', fill_value=0, margins=True))
print('Сумма, млн руб.:')
display((kind_dir.pivot_table(index='kind_label', columns='direction', values='sum_total', aggfunc='sum', fill_value=0, margins=True) / 1e6).round(1))
results['kind_x_direction'] = kind_dir


## 5) Неразмеченное (`kind = other`)

Цифры схлопнуты в `#`. Крупные шаблоны отсюда — кандидаты на новое правило в `KIND_RULES`.
Переводы людям по номеру телефона, скорее всего, будут здесь: в их назначении имена, поэтому шаблоны мелкие.


In [ ]:
other_patterns = num(q(f'''
select
  vid_code,
  direction,
  regexp_replace(c_nazn, '[0-9]+', '#') as nazn_pattern,
  count(*) as docs,
  sum(c_sum) as sum_total,
  min(c_nazn) as nazn_example
from ({CLS}) x
where kind = 'other'
group by 1, 2, 3
order by docs desc
limit 60
'''), ['docs', 'sum_total'])
display(other_patterns)
results['other_patterns'] = other_patterns


## 6) Ордера: что в них

ESPP в банковских, мемориальных и кассовых ордерах — фильтр коллеги `БЕЗН_ПЛ_ПОРУЧ` их отрезает.
Смотрим шаблоны назначений и пары счетов. Если ордера — это `settlement` (30232 ↔ 30102 / 30301), они дублируют
деньги платёжек и в счёт СБП-операций не идут. Если `in` / `out` — это самостоятельные операции.


In [ ]:
ORDER_CODES = ['БЕЗН_БАНК_ОРД', 'БЕЗН_МЕМО_ОРД', 'КАСС_ПРИХОД_ОРД']
in_list = ', '.join(f"'{c}'" for c in ORDER_CODES)

orders_patterns = num(q(f'''
select * from (
  select a.*, row_number() over (partition by vid_code order by docs desc) as rn
  from (
    select
      vid_code, kind, direction,
      regexp_replace(c_nazn, '[0-9]+', '#') as nazn_pattern,
      count(*) as docs,
      sum(c_sum) as sum_total,
      min(c_nazn) as nazn_example
    from ({CLS}) x
    where vid_code in ({in_list})
    group by 1, 2, 3, 4
  ) a
) y
where rn <= 25
order by vid_code, rn
'''), ['docs', 'sum_total'])

orders_accounts = num(q(f'''
select * from (
  select a.*, row_number() over (partition by vid_code order by docs desc) as rn
  from (
    select
      vid_code, bal_dt, bal_kt, direction,
      count(*) as docs,
      sum(c_sum) as sum_total
    from ({CLS}) x
    where vid_code in ({in_list})
    group by 1, 2, 3, 4
  ) a
) y
where rn <= 15
order by vid_code, rn
'''), ['docs', 'sum_total'])

for code_ in ORDER_CODES:
    print(f'\n===== {code_}: шаблоны назначений =====')
    display(orders_patterns[orders_patterns['vid_code'] == code_].drop(columns=['rn']))
    print(f'===== {code_}: пары счетов =====')
    display(orders_accounts[orders_accounts['vid_code'] == code_].drop(columns=['rn']))

results['orders_patterns'] = orders_patterns
results['orders_accounts'] = orders_accounts


## 7) Что оставляет `join eq_account`

По ответу автора скрипта, `eq_account` — подзапрос из ЦТЕ: счета мерчантов
(`z_r2_ip_merchants` → `z_r2_ip_acc.c_dt_1_2` → `z_ac_fin`). Автор пишет, что пока можно без него.

Внутренний join по `c_acc_kt` оставляет только документы, зачисленные **на счёт мерчанта** — то есть замысел скрипта
ближе к СБП-эквайрингу (направление `in`), чем к исходящим платежам клиентов.

Сначала — какие балансовые счета у мерчантов в `eq_account`: если там 30232 / 30233, join оставит и расчётные проводки.
Потом — сколько ESPP-документов проходит join, по видам операций и направлениям.


In [ ]:
ea_bal = num(q(f'''
with {EQ_ACCOUNT_CTE}
select
  substr(c_main_v_id, 1, 5) as bal,
  count(*) as rows_cnt,
  count(distinct id) as accounts
from eq_account
group by 1
order by rows_cnt desc
'''), ['rows_cnt', 'accounts'])
print('Счета мерчантов в eq_account по балансовым счетам:')
display(ea_bal)
dup = ea_bal['rows_cnt'].sum() - ea_bal['accounts'].sum()
if dup > 0:
    print(f'Внимание: {dup:,} лишних строк — один счёт у нескольких мерчантов, inner join коллеги задвоит такие документы')
results['eq_account_bal'] = ea_bal

eq_join = num(q(f'''
with {EQ_ACCOUNT_CTE}
select
  case
    when x.c_acc_kt is null then 'c_acc_kt пуст'
    when ea.id is null then 'нет в eq_account'
    else 'есть в eq_account'
  end as kt_status,
  x.vid_code, x.kind, x.direction, x.bal_kt,
  count(*) as docs,
  sum(x.c_sum) as sum_total
from ({CLS}) x
left join (select distinct id from eq_account) ea on ea.id = x.c_acc_kt
group by 1, 2, 3, 4, 5
order by docs desc
'''), ['docs', 'sum_total'])
eq_join['kind_label'] = eq_join['kind'].map(KIND_LABEL)

print(f'Платёжные поручения ({PAYDOC_CODE}) — что оставляет фильтр коллеги:')
display(add_share(eq_join[eq_join['vid_code'] == PAYDOC_CODE].groupby('kt_status', as_index=False)[['docs', 'sum_total']].sum()))

print('Прошедшие join (все виды документов):')
display(
    eq_join[eq_join['kt_status'] == 'есть в eq_account']
    .groupby(['vid_code', 'kind_label', 'direction', 'bal_kt'], as_index=False)[['docs', 'sum_total']].sum()
    .sort_values('docs', ascending=False)
)
results['eq_account_join'] = eq_join


## 8) Итог для Алины

Сводка: сколько СБП-операций по видам и направлениям, только клиентские направления (`out` / `in`).
В быстром режиме добавлена колонка с грубой оценкой на весь объём (`× 100 / SAMPLE_PCT`).

Ниже печатаются и сохраняются два самостоятельных SQL по `main_docum` с теми же правилами:
- `sbp_espp_final_agg.sql` — агрегат по месяцу, виду документа, виду операции и направлению;
- `sbp_espp_final_detail.sql` — построчно, только СБП и только клиентские направления.

Период в SQL — из ячейки параметров. Запросы читают таблицу целиком, как первый прогон (~10 минут на месяц).


In [ ]:
sbp = kind_dir[kind_dir['is_sbp'] & kind_dir['counted']].copy()
summary = (
    sbp.groupby(['kind', 'kind_label', 'direction', 'direction_label'], as_index=False)[['docs', 'sum_total']]
    .sum()
    .sort_values('docs', ascending=False)
)
if SAMPLE_PCT:
    summary['docs_estimate_full'] = (summary['docs'] * SCALE).round()
    summary['sum_estimate_full'] = (summary['sum_total'] * SCALE).round(2)
display(summary)

tot = kind_dir.groupby(['is_sbp', 'counted'], as_index=False)[['docs', 'sum_total']].sum()
tot['what'] = tot.apply(
    lambda r: ('СБП' if r['is_sbp'] else 'не СБП') + (', клиентские' if r['counted'] else ', расчёты / комиссии / прочее'),
    axis=1,
)
display(tot[['what', 'docs', 'sum_total']])
results['summary_for_alina'] = summary

kinds_in = ', '.join(f"'{k}'" for k in SBP_KINDS)
dirs_in = ', '.join(f"'{d}'" for d in COUNTED_DIRECTIONS)
full_cls = classify_sql(espp_base_sql(None))

FINAL_AGG_SQL = f'''-- СБП в ESPP-документах main_docum, период {PERIOD_START} .. {PERIOD_END_EXCL} (не включая)
-- is_sbp: kind in ({kinds_in}); клиентские направления: direction in ({dirs_in})
select
  substr(d_prov, 1, 7) as month,
  vid_code,
  kind,
  direction,
  count(*) as docs,
  sum(c_sum) as sum_total
from (
{full_cls}
) x
group by 1, 2, 3, 4
order by 1, docs desc'''

FINAL_DETAIL_SQL = f'''-- Построчно: СБП-операции клиентов в ESPP-документах main_docum, период {PERIOD_START} .. {PERIOD_END_EXCL} (не включая)
select *
from (
{full_cls}
) x
where kind in ({kinds_in})
  and direction in ({dirs_in})'''

Path(OUT_SQL_AGG).write_text(FINAL_AGG_SQL, encoding='utf-8')
Path(OUT_SQL_DETAIL).write_text(FINAL_DETAIL_SQL, encoding='utf-8')
print('Сохранено:', Path(OUT_SQL_AGG).resolve(), Path(OUT_SQL_DETAIL).resolve(), sep='\n')
print('\n' + FINAL_AGG_SQL)


## 9) Выгрузка

Каждый результат — отдельный лист Excel. Временная таблица остаётся в `sandbox_ai`; удалить —
`ex(f'drop table if exists {TMP_TABLE}')`.


In [ ]:
with pd.ExcelWriter(OUT_XLSX) as xw:
    for name, df in results.items():
        df.to_excel(xw, sheet_name=name[:31], index=False)
print('Сохранено:', Path(OUT_XLSX).resolve())
print('Листы:', list(results))
